<a href="https://colab.research.google.com/github/davidogm/DataScience/blob/main/ITBD/notebooks/spark/ITBD_W2_Configuracion_S3A_PySpark_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITBD — Semanas 2 y 3
## Configuración : AWS S3 + S3A + PySpark en Colab

Este notebook conserva **el procedimiento con comprobaciones por capas**.

```text
Colab → PySpark/Spark 4.0.4 → Hadoop 3.4.1 → hadoop-aws 3.4.1 + AWS SDK → S3A → S3
```

Objetivo final: leer `s3a://davidgil/trips.csv` directamente con Spark.

## 1. Conceptos
- `s3://` identifica el objeto en Amazon S3.
- Spark/Hadoop accede mediante **S3A**, por eso usamos `s3a://`.
- PySpark por sí solo puede no incluir el conector S3A.
- `boto3` es el SDK Python de AWS; **boto3 y S3A son mecanismos distintos**.

Entorno comprobado: **Spark 4.0.4, Hadoop 3.4.1, hadoop-aws 3.4.1, AWS SDK bundle 2.24.6**.

## 2. Empezar con una sesión limpia
Los JAR deben estar disponibles al crear Spark. Si ya existe una `SparkSession`, usa **Entorno de ejecución → Reiniciar sesión** y ejecuta este notebook desde arriba.

## 3. Instalar librerías Python

In [1]:
!pip -q install boto3 pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.9 MB/s eta 0:00:00


## 4. Descargar dependencias S3A
Añadimos explícitamente `hadoop-aws` y el bundle AWS SDK para controlar el classpath.

In [2]:
!wget -q https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.4.1/hadoop-aws-3.4.1.jar
!wget -q https://repo1.maven.org/maven2/software/amazon/awssdk/bundle/2.24.6/bundle-2.24.6.jar

!ls -lh /content/hadoop-aws-3.4.1.jar /content/bundle-2.24.6.jar

-rw-r--r-- 1 root root 533M Feb 19  2024 /content/bundle-2.24.6.jar
-rw-r--r-- 1 root root 846K Oct  9  2024 /content/hadoop-aws-3.4.1.jar


## 5. Credenciales temporales del AWS Lab
Necesitamos **Access Key + Secret Key + Session Token**. No guardes valores reales en el notebook ni los compartas.

In [3]:
import os, getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass.getpass("AWS_ACCESS_KEY_ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
os.environ["AWS_SESSION_TOKEN"] = getpass.getpass("AWS_SESSION_TOKEN: ")
print("✅ Credenciales cargadas en esta sesión")

AWS_ACCESS_KEY_ID: ··········
AWS_SECRET_ACCESS_KEY: ··········
AWS_SESSION_TOKEN: ··········
✅ Credenciales cargadas en esta sesión


## 6. Comprobación 1 — autenticación
STS responde a **¿quién soy?**. Que funcione prueba autenticación, no permiso sobre S3.

In [4]:
import boto3
boto3.DEFAULT_SESSION = None
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print("✅ Autenticación correcta")
print("Account:", identity["Account"])
print("Arn:", identity["Arn"])

✅ Autenticación correcta
Account: 713754066836
Arn: arn:aws:sts::713754066836:assumed-role/voclabs/user5495267=_Student_View__David_Gil


## 7. Comprobación 2 — autorización `GetObject`
Probamos la operación que realmente necesitamos. Mantén el bucket privado.

In [5]:
from botocore.exceptions import ClientError
s3 = boto3.client("s3")

try:
    obj = s3.get_object(Bucket="davidgil", Key="trips.csv")
    print("✅ s3:GetObject permitido")
    print("Tamaño:", obj["ContentLength"], "bytes")
except ClientError as e:
    print("❌ Código:", e.response["Error"]["Code"])
    print("Mensaje:", e.response["Error"]["Message"])

✅ s3:GetObject permitido
Tamaño: 111082 bytes


### Caso AWS Lab: `voc-cancel-cred`
En nuestras pruebas apareció un `explicit deny` de `voc-cancel-cred` cuando el Lab estaba apagado. Reiniciar correctamente el laboratorio restauró el acceso.

Si aparece: comprobar Lab **Running** → reiniciar si procede → revisar credenciales → recrear clientes boto3 → repetir `GetObject`. **No hacer público el bucket.**

## 8. Crear Spark con los JAR S3A

In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITBD-S3")
    .master("local[*]")
    .config("spark.jars",
            "/content/hadoop-aws-3.4.1.jar,/content/bundle-2.24.6.jar")
    .getOrCreate()
)

print("Spark:", spark.version)
jvm = spark.sparkContext._jvm
print("Hadoop:", jvm.org.apache.hadoop.util.VersionInfo.getVersion())

Spark: 4.0.4
Hadoop: 3.4.1


## 9. Comprobación 3 — JAR registrados por Spark

In [7]:
print("JARs configurados:")
print(spark.sparkContext.getConf().get("spark.jars"))
print("\nJARs conocidos por Spark:")
print(spark.sparkContext._jsc.sc().listJars())

JARs configurados:
/content/hadoop-aws-3.4.1.jar,/content/bundle-2.24.6.jar

JARs conocidos por Spark:
List(spark://82a1b7ff49c0:36709/jars/bundle-2.24.6.jar, spark://82a1b7ff49c0:36709/jars/hadoop-aws-3.4.1.jar)


Deben aparecer `hadoop-aws-3.4.1.jar` y `bundle-2.24.6.jar`.

## 10. Comprobación 4 — Hadoop reconoce S3A
Esta es la prueba importante. Pedimos directamente a Hadoop un `FileSystem` para `s3a://`.

Durante el diagnóstico, `Class.forName()` dio un negativo engañoso por el *classloader*, mientras Hadoop sí podía crear `S3AFileSystem`; por eso no usamos esa comprobación en el procedimiento definitivo.

In [8]:
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

try:
    uri = spark.sparkContext._jvm.java.net.URI("s3a://davidgil")
    fs = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(uri, hadoop_conf)
    print("✅ Hadoop reconoce S3A")
    print("FileSystem:", fs.getClass().getName())
except Exception as e:
    print("❌ Hadoop no puede crear S3A")
    print(e)

✅ Hadoop reconoce S3A
FileSystem: org.apache.hadoop.fs.s3a.S3AFileSystem


Resultado esperado:
```text
✅ Hadoop reconoce S3A
FileSystem: org.apache.hadoop.fs.s3a.S3AFileSystem
```

## 11. Pasar las credenciales temporales a Hadoop S3A
Que boto3 tenga credenciales no implica que S3A las esté usando. Configuramos el proveedor de credenciales temporales.

In [9]:
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
hadoop_conf.set("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
hadoop_conf.set("fs.s3a.session.token", os.environ["AWS_SESSION_TOKEN"])
hadoop_conf.set("fs.s3a.aws.credentials.provider",
                "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")

print("✅ Credenciales temporales configuradas para S3A")

✅ Credenciales temporales configuradas para S3A


## 12. Prueba final — Spark lee directamente desde S3

In [10]:
s3_path = "s3a://davidgil/trips.csv"

trips_s3 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(s3_path)
)

trips_s3.show(5)

+---+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-------------+-------------+
|_c0|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee19|airport_fee20|
+---+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-------------+-------------+
|  0|       2| 2023-06-30 23:59:59|  2023-07-01 00:47:49|            2.0|        17.62|       2.0|                 N|         1

## 13. Comprobaciones finales

In [11]:
trips_s3.printSchema()
print("Particiones:", trips_s3.rdd.getNumPartitions())
print("Número de viajes:", trips_s3.count())

root
 |-- _c0: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee19: double (nullable = true)
 |-- airport_fee20: string (nullable = true)

Particiones: 1
Número de viajes: 1000


## 14. Diagnóstico por capas

| Error | Capa probable | Comprobar |
|---|---|---|
| `ModuleNotFoundError: boto3` | Python | instalar boto3 |
| STS falla | autenticación | credenciales temporales |
| `403` / `AccessDenied` | autorización | Lab activo / IAM / GetObject |
| `voc-cancel-cred` | AWS Lab | reiniciar Lab y revisar credenciales |
| `S3AFileSystem not found` | Hadoop/Spark | hadoop-aws / classpath |
| JAR no aparece en `listJars()` | Spark | crear sesión limpia con `spark.jars` |
| S3A existe pero lectura falla | credenciales/permisos | `fs.s3a.*` y AWS Lab |

Orden recomendado:
```text
STS → boto3 GetObject → versiones Spark/Hadoop → JARs → S3A → credenciales S3A → spark.read
```

## 15. Resumen
```text
Python → boto3 → AWS S3

Spark → Hadoop → S3A → AWS S3
```
Son caminos distintos. Cuando `trips_s3.show(5)` funciona, la infraestructura está lista y el foco puede pasar al procesamiento con Spark.

### Seguridad
Mantén el bucket privado, no publiques credenciales y recuerda que las credenciales del AWS Lab son temporales.